# Portfolio edition
Nicholas Koch | Sports analytics

Saved outputs and execution metadata have been cleared. Optional odds requests default to off. Model logic is unchanged. See the project README for methods, limitations, and setup. Run in Google Colab to generate current results; this copy has not been executed end to end.


# Nicolay’s NHL Points Model
**Daily probability of at least one point (goal + assist), conditional on playing.**

### Quick start
1. Upload this notebook to Google Colab. Use a normal CPU runtime; a GPU is unnecessary.
2. Set `TARGET_DATE` below, or leave it blank for today in New York time.
3. Choose **Runtime → Run all**. Approve Google Drive access if you want your cache and results to persist.
4. Read the test results, then the daily rankings. CSVs are saved automatically.

The first run downloads several seasons and will be the slowest. Progress prints by month and page. Later runs reuse the historical cache and saved models, refreshing recent data. NHL game data needs no API key. The optional automatic odds feed in Section 11 needs a one-time provider API key and is subject to its quota.

**Scope:** regular-season skaters only. The model uses lagged scoring, shots, total and power-play ice time, team offense, opponent defense, home ice, and rest. It combines logistic regression and gradient-boosted trees, then calibrates probabilities on a later period. A separate Poisson model estimates expected points.

**What is not included:** confirmed lines, injuries, confirmed starting goalies, or betting prices in the predictive model itself. Section 11 compares external odds after prediction. Current rosters are candidate lists, not confirmed game lineups. Every prediction is conditional on the player actually playing. Recent ice time proxies for role; it is not a live line assignment. No profit or accuracy level is promised.

**Validation status at delivery:** code and offline fixtures were checked. This authoring environment could not access the NHL feeds, so no real-data accuracy, calibration, or profitability has been established. The notebook performs that backtest when you run it in Colab. It stops on missing required fields or incomplete downloads instead of substituting made-up data.

# Update: automatic evaluation and simulated results
Run **Runtime → Run all** as usual, with Google Drive enabled. No bet entry is needed.

- Section 12 reports actual saved prediction performance by probability range and date.
- Section 13 freezes one simulated selection set per forecast date, on the first run with usable sportsbook quotes. It grades those selections on subsequent runs. Reruns never replace a frozen set, including a zero-selection set.
- Simulation uses the best qualifying saved sportsbook price per player, 1 unit per straight, at least 3% estimated EV, positive EV after a 3 percentage-point buffer, at least 10 prior games, and no long-layoff flag. These research rules are not proven profitable. Playing status remains unconfirmed; DNP is refunded in this simulation.
- Existing predictions and history are reused. Simulation starts when you run this update; prior bets are not reconstructed. It does not track personal profit or Underdog entries.
- Prediction models and calibration logic are unchanged. This update improves monitoring, not demonstrated accuracy. The first run refits once because cache identity now includes model settings.
- All outputs remain in your existing Google Drive model folder. Reports are under `evaluation`; simulated results are under `evaluation/paper/<rules ID>`.

Changing PAPER_RULES starts a separate strategy history. Routine runs require no setting changes. Run after prior games are final and before today's games start; run all again on a later day to settle results.


## 1. Settings
For your normal daily run, only change the date and any players or teams you want to exclude. Leave the other settings alone initially. Past-date roster predictions are disabled; the historical evaluation below is the proper way to test old games.

In [ ]:
TARGET_DATE = ""                 # Example: "2026-10-15". Blank = today (New York).
START_SEASON = 2023               # 2023 means the 2023–24 season.
USE_GOOGLE_DRIVE = True           # Recommended: preserves downloads between Colab sessions.
EXCLUDE_PLAYERS = []              # Exact full names, e.g. ["Connor McDavid"]. Case-insensitive.
EXCLUDE_TEAMS = []                # Team abbreviations, e.g. ["TOR"].
TOP_N = 30
MIN_PRIOR_GAMES = 10              # Less history is still scored, but flagged.
FORCE_RETRAIN = False             # Normally unnecessary; new data triggers retraining.
FORCE_DATA_REFRESH = False        # True re-downloads all data; normally leave False.
# Optional: only rank these player IDs (after you verify the expected lineup).
ONLY_PLAYER_IDS = []
MODEL_VERSION = "nhl_points_v1.0"

# ODDS BUDGET SETTINGS — independent of historical-data/model refresh settings.
FETCH_ODDS = False  # Enable only after configuring your own API key.
ODDS_REFRESH_TOKEN = ""   # Normal: leave blank. To refresh once, enter a NEW label, e.g. "pregame-1".
# Reusing the same label reuses that day's cache. Clear it after refreshing.
ODDS_CREDIT_RESERVE = 25  # Stop new paid requests before the account falls below this.
ODDS_MAX_CREDITS_PER_RUN = 20


## 2. Setup and storage
Google Colab already includes the scientific Python packages used here. Outside Colab, install `numpy pandas scikit-learn matplotlib joblib` if needed. If Drive cannot mount, the notebook asks you to fix it or explicitly turn Drive off, rather than silently losing persistent storage.

In [ ]:
import json, hashlib, time, sys, warnings, urllib.request, urllib.parse, urllib.error
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, mean_absolute_error
from sklearn.calibration import calibration_curve
try:
    from IPython.display import display
except ImportError:
    display = print

TODAY = pd.Timestamp(datetime.now(ZoneInfo("America/New_York")).date())
TARGET = pd.Timestamp(TARGET_DATE or TODAY.date()).normalize()
if TARGET < TODAY:
    raise ValueError("Use today or a future date for live rankings. Current rosters cannot reconstruct historical lineups.")
if TARGET > TODAY + pd.Timedelta(days=7):
    raise ValueError("Choose a date within 7 days. Rosters and player roles change.")
CUTOFF = min(TARGET, TODAY)          # Only games strictly before this date enter training.
if pd.Timestamp(f"{START_SEASON}-09-01") >= CUTOFF:
    raise ValueError("START_SEASON must precede the forecast date.")

IN_COLAB = "google.colab" in sys.modules
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    mount = Path("/content/drive")
    if not (mount / "MyDrive").is_dir():
        drive.mount(str(mount))
    ROOT = mount / "MyDrive" / "Nicolay_NHL_Points_Model"
else:
    ROOT = Path("/content/Nicolay_NHL_Points_Model" if IN_COLAB else "Nicolay_NHL_Points_Model")
    if USE_GOOGLE_DRIVE:
        print("Outside Colab: using a local folder.")
    elif IN_COLAB:
        print("Drive is OFF: files will be lost if this Colab runtime resets. Download anything you need.")
for folder in [ROOT, ROOT / "cache", ROOT / "models", ROOT / "predictions", ROOT / "evaluation"]:
    folder.mkdir(parents=True, exist_ok=True)
print("Forecast:", TARGET.date(), "| Historical cutoff (exclusive):", CUTOFF.date())
print("Files:", ROOT.resolve())

## 3. NHL download helpers
Repeat runs load one compressed history file. Completed months also have compact caches; recent data refreshes after 18 hours. The first run of this update may still take time to consolidate existing page caches. Messages distinguish **CACHED HISTORY**, **CACHED MONTH**, and **NETWORK**.

Historical player-game records come from NHL Stats REST. Schedule and current roster data come from NHL Gamecenter. Requests are paginated, retried, cached, and checked for duplicates and missing fields. A failed download stops the run; rerunning resumes cached pages. Data from the most recent month is refreshed daily. Set `FORCE_DATA_REFRESH=True` if you need older statistical corrections.

In [ ]:
import gzip

CACHE_COUNTS = {"pages_read": 0, "downloads": 0}

STATS = "https://api.nhle.com/stats/rest/en"
WEB = "https://api-web.nhle.com/v1"

def atomic_json(path, value):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

def get_json(url, params=None, ttl_hours=None):
    if params:
        url += "?" + urllib.parse.urlencode(params)
    key = hashlib.sha256(url.encode()).hexdigest()
    path = ROOT / "cache" / (key + ".json")
    fresh = path.exists() and (ttl_hours is None or time.time() - path.stat().st_mtime < ttl_hours * 3600)
    if fresh and not FORCE_DATA_REFRESH:
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
            CACHE_COUNTS["pages_read"] += 1
            return value
        except (ValueError, OSError):
            pass
    last_error = None
    for attempt in range(4):
        try:
            CACHE_COUNTS["downloads"] += 1
            if CACHE_COUNTS["downloads"] == 1 or CACHE_COUNTS["downloads"] % 25 == 0:
                print(f'  NETWORK: {CACHE_COUNTS["downloads"]} requests this run', flush=True)
            request = urllib.request.Request(url, headers={"User-Agent": "NicolayNHLModel/1.0", "Accept": "application/json"})
            with urllib.request.urlopen(request, timeout=60) as response:
                data = json.load(response)
            if not isinstance(data, dict):
                raise ValueError("Expected a JSON object.")
            atomic_json(path, data)
            time.sleep(0.15)  # Keep sequential requests modest.
            return data
        except (urllib.error.URLError, TimeoutError, ValueError, OSError) as err:
            last_error = err
            if isinstance(err, urllib.error.HTTPError) and err.code in [400, 401, 403, 404]:
                break
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"NHL download failed: {last_error}. Retry later in Colab. URL: {url}")

def _report_month_pages(report, begin, end):
    rows, offset, expected = [], 0, None
    # Recent records may still receive corrections; old pages can stay cached.
    ttl = 18 if end >= TODAY - pd.Timedelta(days=35) else None
    while True:
        params = {
            "isAggregate": "false", "isGame": "true", "start": offset, "limit": 1000,
            "sort": json.dumps([{"property": "gameId", "direction": "ASC"},
                                {"property": "playerId", "direction": "ASC"}]),
            "cayenneExp": f'gameTypeId=2 and gameDate>="{begin:%Y-%m-%d}" and gameDate<="{end:%Y-%m-%d} 23:59:59"',
        }
        data = get_json(f"{STATS}/skater/{report}", params, ttl)
        if "data" not in data or "total" not in data:
            raise ValueError(f"Unexpected {report} response; data/total missing.")
        batch, total = data["data"], int(data["total"])
        if expected is None:
            expected = total
        elif total != expected:
            raise RuntimeError("The NHL changed this report during pagination. Refresh data and rerun.")
        if not batch:
            if offset != expected:
                raise RuntimeError(f"Incomplete {report} download: {offset}/{expected}.")
            break
        rows.extend(batch)
        offset += len(batch)         # Works even when the server caps the requested page size.
        if offset >= expected:
            break
        if offset % 5000 < len(batch):
            print(f"  {report}: {offset:,}/{expected:,}", flush=True)
    if len(rows) != expected:
        raise RuntimeError(f"Wrong row count for {report}: {len(rows)} != {expected}.")
    return rows

def read_bundle(path, ttl_hours):
    if FORCE_DATA_REFRESH:
        return None
    try:
        if ttl_hours is not None and time.time() - path.stat().st_mtime >= ttl_hours * 3600:
            return None
        with gzip.open(path, "rt", encoding="utf-8") as stream:
            value = json.load(stream)
        if value.get("cache_schema") != 1:
            return None
        return value
    except (OSError, EOFError, ValueError):
        return None

def write_bundle(path, value):
    value = dict(value, cache_schema=1)
    temp = path.with_suffix(path.suffix + ".tmp")
    with gzip.open(temp, "wt", encoding="utf-8", compresslevel=1) as stream:
        json.dump(value, stream, ensure_ascii=False)
    temp.replace(path)

def report_month(report, begin, end):
    path = ROOT / "cache" / f"month_v1_{report}_{begin:%Y%m%d}_{end:%Y%m%d}.json.gz"
    ttl = 18 if end >= TODAY - pd.Timedelta(days=35) else None
    cached = read_bundle(path, ttl)
    if cached is not None:
        print(f"  CACHED MONTH: {report}, {len(cached['rows']):,} rows", flush=True)
        return cached["rows"]
    old_reads, old_downloads = CACHE_COUNTS["pages_read"], CACHE_COUNTS["downloads"]
    print(f"  Preparing compact {report} cache; reusing existing pages where available...", flush=True)
    rows = _report_month_pages(report, begin, end)
    write_bundle(path, {"rows": rows})
    print(f"  {CACHE_COUNTS['pages_read'] - old_reads} cached pages read; "
          f"{CACHE_COUNTS['downloads'] - old_downloads} network requests. Compact cache saved.", flush=True)
    return rows

def load_raw_history():
    bundle_path = ROOT / "cache" / f"history_v1_{START_SEASON}_{CUTOFF:%Y%m%d}.json.gz"
    cached = read_bundle(bundle_path, ttl_hours=18)
    if cached is not None:
        print("CACHED HISTORY: reading one compact file. No historical downloads needed.", flush=True)
        return pd.DataFrame(cached["summary"]), pd.DataFrame(cached["ice"])
    print("Building/updating history. Completed months will use compact caches.", flush=True)
    summary, ice = [], []
    start = pd.Timestamp(f"{START_SEASON}-09-01")
    for begin in pd.date_range(start, CUTOFF - pd.Timedelta(days=1), freq="MS"):
        # No NHL regular-season games in July or August in the supported modern sample.
        if begin.month in [7, 8]:
            continue
        end = min(begin + pd.offsets.MonthEnd(0), CUTOFF - pd.Timedelta(days=1))
        print(f"Loading {begin:%Y-%m} through {end:%Y-%m-%d} ...", flush=True)
        part = report_month("summary", begin, end)
        summary.extend(part)
        if part:
            ice.extend(report_month("timeonice", begin, end))
        print(f"  {len(part):,} player-games", flush=True)
    if not summary:
        raise RuntimeError("No regular-season player-game records returned. Check START_SEASON and API access.")
    write_bundle(bundle_path, {"summary": summary, "ice": ice})
    print("Compact history saved: same-date reruns can now load one file.", flush=True)
    return pd.DataFrame(summary), pd.DataFrame(ice)

def field(frame, *names):
    for name in names:
        if name in frame.columns:
            return frame[name]
    raise ValueError(f"Required NHL field missing: {names}. Available fields: {list(frame.columns)}")

def minutes(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str) and ":" in value:
        parts = [float(x) for x in value.split(":")]
        if len(parts) != 2:
            raise ValueError(f"Unexpected ice-time format: {value}")
        return parts[0] + parts[1] / 60
    return float(value) / 60         # NHL numeric time values are seconds.

def normalize_history(summary, ice):
    for frame in [summary, ice]:
        if frame.empty or not {"playerId", "gameId"}.issubset(frame.columns):
            raise ValueError("Missing playerId/gameId or empty ice-time report.")
        if frame.duplicated(["playerId", "gameId"]).any():
            raise ValueError("Duplicate player-game records from NHL; refusing to train.")
    h = pd.DataFrame({
        "player_id": field(summary, "playerId").astype(int),
        "game_id": field(summary, "gameId").astype(int),
        "date": pd.to_datetime(field(summary, "gameDate").astype(str).str[:10]),
        "player": field(summary, "skaterFullName"),
        "team": field(summary, "teamAbbrev", "teamAbbrevs"),
        "opponent": field(summary, "opponentTeamAbbrev", "opponentTeamAbbrevs"),
        "position": field(summary, "positionCode", "playerPositionCode"),
        "home": field(summary, "homeRoad").eq("H").astype(int),
    })
    for name, source in {"goals": "goals", "assists": "assists", "points": "points", "shots": "shots", "pp_points": "ppPoints"}.items():
        h[name] = pd.to_numeric(field(summary, source), errors="raise")
    h["toi"] = field(summary, "timeOnIcePerGame", "timeOnIce").map(minutes)
    pp = pd.DataFrame({
        "player_id": ice["playerId"].astype(int), "game_id": ice["gameId"].astype(int),
        "pp_toi": field(ice, "ppTimeOnIcePerGame", "ppTimeOnIce").map(minutes),
    })
    h = h.merge(pp, on=["player_id", "game_id"], how="left", validate="one_to_one")
    if h["pp_toi"].isna().mean() > 0.01:
        raise ValueError("More than 1% of player-game records lack PP ice time. Check NHL report fields/coverage.")
    if not h["home"].isin([0, 1]).all() or not field(summary, "homeRoad").isin(["H", "R"]).all():
        raise ValueError("Unexpected home/road coding.")
    if h[["goals", "assists", "points", "shots", "toi"]].isna().any().any():
        raise ValueError("Missing required game outcomes or ice time.")
    if not (h["points"] == h["goals"] + h["assists"]).all():
        raise ValueError("Points must equal goals plus assists.")
    if not h["toi"].between(0, 90).all() or not (h["pp_toi"].fillna(0) <= h["toi"] + 0.02).all():
        raise ValueError("Ice-time values failed unit checks.")
    if not (h["date"] < CUTOFF).all():
        raise ValueError("Historical data contains forecast-day or future outcomes.")
    if not h["team"].astype(str).str.fullmatch(r"[A-Z]{2,3}").all():
        raise ValueError("Expected individual-game team abbreviations.")
    # Previous-calendar-day games can still be in progress just after midnight.
    recent = h["date"] >= CUTOFF - pd.Timedelta(days=7)
    if recent.any():
        schedule = get_json(f"{WEB}/schedule/{CUTOFF - pd.Timedelta(days=7):%Y-%m-%d}", ttl_hours=0)
        if "gameWeek" not in schedule:
            raise ValueError("Cannot verify that recent games have finished.")
        games = {int(g["id"]): g for day in schedule["gameWeek"] for g in day.get("games", [])}
        if not set(h.loc[recent, "game_id"]).issubset(games):
            raise ValueError("Recent stats games are missing from the schedule. Retry after NHL feeds update.")
        final_ids = {gid for gid, g in games.items() if g.get("gameState") in ["FINAL", "OFF"]}
        complete = ~recent | h["game_id"].isin(final_ids)
        if not complete.all():
            print("Excluded unfinished recent player-game rows:", int((~complete).sum()))
        h = h[complete].copy()
    h = h[h["toi"] > 0].copy()       # Scratches are not zero-point appearances.
    h["hit"] = (h["points"] >= 1).astype(int)
    return h.sort_values(["date", "game_id", "player_id"]).reset_index(drop=True)

## 4. Download and validate history
This loads the compact history cache when available. **Preparing compact cache** means this update is consolidating the old page cache once. **NETWORK** messages identify actual requests. Keep the tab open. If the NHL temporarily rejects a request, wait and rerun; cached successful pages remain available.

In [ ]:
raw_summary, raw_ice = load_raw_history()
history = normalize_history(raw_summary, raw_ice)
print(f"\n{len(history):,} player-games | {history.player_id.nunique():,} skaters")
print("Date range:", history.date.min().date(), "to", history.date.max().date())
print("Power-play ice-time coverage:", f"{history.pp_toi.notna().mean():.1%}")
display(history.tail(5))

## 5. Build features using earlier games only
Each rolling statistic is shifted by one game. Team statistics are calculated once per team-game, not once per player. All games on a date stay together in the train/calibration/test split. Teams and players retain their prior history across seasons; long layoffs are explicitly recorded. The ARI → UTA relocation is linked for team history.

Team offense/defense features sum **skater** goals and shots from the downloaded games, then pair opponents. They exclude rare goalie goals and shootout deciding goals. Historical scratches are excluded; the target is conditional on an appearance.

In [ ]:
PLAYER_STATS = ["points", "goals", "assists", "shots", "toi", "pp_toi", "pp_points", "hit"]
WINDOWS = [5, 20, 60]
TEAM_STATS = ["skater_goals", "skater_shots", "skater_goals_allowed", "skater_shots_allowed"]
FEATURES = ([f"{x}_last{w}" for x in PLAYER_STATS for w in WINDOWS]
            + ["prior_games", "days_since_played", "home", "is_defense", "is_center", "is_wing"]
            + [f"{side}_{x}_last20" for side in ["team", "opp"] for x in TEAM_STATS]
            + ["team_rest", "opp_rest"])

def team_key(series):
    return series.replace({"ARI": "UTA"})

def team_game_rows(h):
    team = h.groupby(["game_id", "date", "team", "opponent"], as_index=False).agg(
        skater_goals=("goals", "sum"), skater_shots=("shots", "sum"))
    if not team.groupby("game_id").size().eq(2).all():
        raise ValueError("A game is missing one team's skater data.")
    other = team[["game_id", "team", "skater_goals", "skater_shots"]].rename(columns={
        "team": "opponent", "skater_goals": "skater_goals_allowed", "skater_shots": "skater_shots_allowed"})
    team = team.merge(other, on=["game_id", "opponent"], validate="one_to_one")
    return team

def build_features(h, future=None):
    h = h.copy()
    h["forecast"] = False
    if future is not None and len(future):
        future = future.copy()
        future["forecast"] = True
        if future["date"].min() <= h["date"].max():
            raise ValueError("Forecast rows must follow the historical data.")
        for col in PLAYER_STATS:
            future[col] = np.nan
        d = pd.concat([h, future], ignore_index=True)
    else:
        d = h.copy()
    d = d.sort_values(["player_id", "date", "game_id"]).reset_index(drop=True)
    group = d.groupby("player_id", sort=False)
    d["prior_games"] = group.cumcount()
    d["days_since_played"] = group["date"].diff().dt.days.clip(upper=180)
    for stat in PLAYER_STATS:
        for window in WINDOWS:
            d[f"{stat}_last{window}"] = group[stat].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    d["is_defense"] = d["position"].eq("D").astype(int)
    d["is_center"] = d["position"].eq("C").astype(int)
    d["is_wing"] = d["position"].isin(["L", "R", "LW", "RW"]).astype(int)
    # Smoothed recent hit-rate benchmark; fixed prior is a baseline assumption, not future data.
    n = d["prior_games"].clip(upper=60)
    d["baseline_probability"] = (d["hit_last60"].fillna(0) * n + 8 * 0.4) / (n + 8)
    t = team_game_rows(h)
    if future is not None and len(future):
        new_t = future[["game_id", "date", "team", "opponent"]].drop_duplicates()
        # Include both teams even if one team's players were excluded from ranking.
        reverse = new_t.rename(columns={"team": "opponent", "opponent": "team"})
        new_t = pd.concat([new_t, reverse]).drop_duplicates(["game_id", "team"])
        t = pd.concat([t, new_t], ignore_index=True)
    t["team_key"] = team_key(t["team"])
    t = t.sort_values(["team_key", "date", "game_id"])
    tg = t.groupby("team_key", sort=False)
    t["rest"] = tg["date"].diff().dt.days.clip(upper=30)
    for stat in TEAM_STATS:
        t[f"{stat}_last20"] = tg[stat].transform(lambda s: s.shift(1).rolling(20, min_periods=1).mean())
    context = [f"{x}_last20" for x in TEAM_STATS] + ["rest"]
    own = t[["game_id", "team"] + context].rename(columns={x: f"team_{x}" for x in context})
    opp = t[["game_id", "team"] + context].rename(columns={"team": "opponent", **{x: f"opp_{x}" for x in context}})
    d = d.merge(own, on=["game_id", "team"], how="left", validate="many_to_one")
    d = d.merge(opp, on=["game_id", "opponent"], how="left", validate="many_to_one")
    return d.sort_values(["date", "game_id", "player_id"]).reset_index(drop=True)

featured = build_features(history)
# Skip the initial 30 calendar days so the benchmark/features have a warm-up period.
model_data = featured[featured.date >= featured.date.min() + pd.Timedelta(days=30)].copy()
print("Model rows:", f"{len(model_data):,}", "| Features:", len(FEATURES))

## 6. Train, calibrate, and test
The earliest 65% of game dates train the models, the next 15% calibrate the point probability, and the last 20% form an untouched test period. Earlier outcomes within that test period can update rolling features, as they would in daily use; model weights remain fixed throughout the test.

The classifier uses a fixed 50/50 blend of logistic regression and gradient boosting. A logistic calibrator learns on later games only. The expected-points model is separate, so its mean and the 1+ point probability need not imply the same distribution.

**Lower Brier score and log loss are better.** Calibration compares predicted probabilities with observed point rates. The table includes both a constant training-average baseline and a player's smoothed recent hit rate. Do not assume the model adds value unless it beats these benchmarks. This backtest covers players who actually played, not historical roster/injury decisions, and it does not measure betting returns.

In [ ]:
def new_models():
    linear = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                           LogisticRegression(C=0.15, max_iter=1200))
    trees = HistGradientBoostingClassifier(max_iter=180, max_leaf_nodes=15, min_samples_leaf=100,
                                          learning_rate=0.05, l2_regularization=10,
                                          early_stopping=False, random_state=42)
    count = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                          PoissonRegressor(alpha=0.1, max_iter=1000))
    return linear, trees, count

def probability_logit(p):
    p = np.clip(np.asarray(p), 0.0001, 0.9999)
    return np.log(p / (1 - p)).reshape(-1, 1)

def raw_probability(bundle, frame):
    x = frame[FEATURES]
    return (bundle["linear"].predict_proba(x)[:, 1] + bundle["trees"].predict_proba(x)[:, 1]) / 2

def fit_bundle(train, calibration):
    if train.hit.nunique() != 2 or calibration.hit.nunique() != 2:
        raise ValueError("Both outcomes are required in training and calibration.")
    linear, trees, count = new_models()
    linear.fit(train[FEATURES], train.hit.astype(int))
    trees.fit(train[FEATURES], train.hit.astype(int))
    count.fit(train[FEATURES], train.points)
    bundle = {"linear": linear, "trees": trees, "count": count}
    calibrator = LogisticRegression(C=1.0, max_iter=1000)
    calibrator.fit(probability_logit(raw_probability(bundle, calibration)), calibration.hit.astype(int))
    if calibrator.coef_[0, 0] <= 0:
        raise RuntimeError("Calibration reversed the ranking. Investigate the data/model before using it.")
    bundle["calibrator"] = calibrator
    return bundle

def predict_bundle(bundle, frame):
    p = bundle["calibrator"].predict_proba(probability_logit(raw_probability(bundle, frame)))[:, 1]
    expected = bundle["count"].predict(frame[FEATURES])
    return np.clip(p, 0.0001, 0.9999), np.maximum(expected, 0)

def metric_row(name, y, p):
    return {"Model": name, "Brier": brier_score_loss(y, p), "Log loss": log_loss(y, p, labels=[0, 1]),
            "AUC": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan}

# Include estimator settings and calibration/split policy in cache identity.
# Bump MODEL_VERSION if editing feature calculations or other model logic.
model_signature = json.dumps({
    "estimators": [m.get_params(deep=True) for m in new_models()],
    "calibrator": LogisticRegression(C=1.0, max_iter=1000).get_params(),
    "blend": [0.5, 0.5], "live_calibration_dates": 45,
    "test_split": [0.65, 0.80], "cache_schema": 2
}, sort_keys=True, default=str)
# Include exact downloaded data and library version in cache identity.
fingerprint = hashlib.sha256(pd.util.hash_pandas_object(history, index=False).values.tobytes()
                             + (MODEL_VERSION + sklearn.__version__ + json.dumps(FEATURES) + model_signature).encode()).hexdigest()[:16]
model_path = ROOT / "models" / f"model_{fingerprint}.joblib"
if model_path.exists() and not FORCE_RETRAIN:
    saved = joblib.load(model_path)  # Only load files created by your own notebook.
    live_bundle, evaluation, reliability, test_predictions = [saved[k] for k in ["live_bundle", "evaluation", "reliability", "test_predictions"]]
    print("Loaded cached model and its original backtest.")
else:
    dates = np.sort(model_data.date.unique())
    if len(dates) < 120:
        raise ValueError("Need at least 120 game dates. Use more historical seasons.")
    cut1, cut2 = dates[int(len(dates) * 0.65)], dates[int(len(dates) * 0.80)]
    train = model_data[model_data.date < cut1]
    calibration = model_data[(model_data.date >= cut1) & (model_data.date < cut2)]
    test = model_data[model_data.date >= cut2]
    for label, frame in [("Train", train), ("Calibrate", calibration), ("Test", test)]:
        print(f"{label}: {frame.date.min().date()}–{frame.date.max().date()} | {len(frame):,} rows")
    test_bundle = fit_bundle(train, calibration)
    p, expected = predict_bundle(test_bundle, test)
    baseline = test.baseline_probability.to_numpy()
    evaluation = pd.DataFrame([
        metric_row("Constant training rate", test.hit, np.full(len(test), train.hit.mean())),
        metric_row("Recent player hit rate", test.hit, baseline),
        metric_row("Calibrated model", test.hit, p),
    ])
    test_predictions = test[["date", "game_id", "player_id", "player", "team", "opponent", "hit", "points", "prior_games"]].copy()
    test_predictions["probability"] = p
    test_predictions["baseline_probability"] = baseline
    test_predictions["expected_points"] = expected
    test_predictions["baseline_expected_points"] = test.points_last60.fillna(train.points.mean())
    test_predictions["probability_bin"] = pd.cut(p, np.linspace(0, 1, 11), include_lowest=True)
    reliability = test_predictions.groupby("probability_bin", observed=True).agg(
        predictions=("hit", "size"), mean_prediction=("probability", "mean"), actual_rate=("hit", "mean")).reset_index()
    print("Expected-points test MAE:", round(mean_absolute_error(test.points, expected), 4))
    print("Recent-mean baseline MAE:", round(mean_absolute_error(test.points, test_predictions.baseline_expected_points), 4))
    # For live predictions, use all available dates while reserving the last ~45 game dates for calibration.
    live_cut = dates[-45]
    print("Fitting daily model; calibration begins", pd.Timestamp(live_cut).date())
    live_bundle = fit_bundle(model_data[model_data.date < live_cut], model_data[model_data.date >= live_cut])
    saved = {"live_bundle": live_bundle, "evaluation": evaluation, "reliability": reliability,
             "test_predictions": test_predictions, "features": FEATURES,
             "history_through": str(history.date.max().date()), "fingerprint": fingerprint,
             "model_version": MODEL_VERSION, "sklearn_version": sklearn.__version__}
    temp = model_path.with_suffix(".tmp")
    joblib.dump(saved, temp)
    temp.replace(model_path)
    evaluation.to_csv(ROOT / "evaluation" / f"metrics_{fingerprint}.csv", index=False)
    reliability.to_csv(ROOT / "evaluation" / f"calibration_{fingerprint}.csv", index=False)
    test_predictions.to_csv(ROOT / "evaluation" / f"test_predictions_{fingerprint}.csv", index=False)

display(evaluation.round(4))
display(reliability)
model_brier = float(evaluation.loc[evaluation.Model == "Calibrated model", "Brier"].iloc[0])
baseline_brier = float(evaluation.loc[evaluation.Model == "Recent player hit rate", "Brier"].iloc[0])
if model_brier >= baseline_brier:
    print("CAUTION: the model did not beat the recent-player baseline on test Brier score. Treat rankings as experimental.")
print("Test dates:", str(test_predictions.date.min())[:10], "to", str(test_predictions.date.max())[:10])
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(reliability.mean_prediction, reliability.actual_rate, "o-", label="Untouched test games")
ax.set(xlabel="Predicted chance of 1+ point", ylabel="Actual fraction with 1+ point", title="NHL points model: calibration", xlim=(0, 1), ylim=(0, 1))
ax.legend(); ax.grid(alpha=0.2); plt.show()

## 7. Get the selected slate and current roster candidates
Only regular-season games that have not started are ranked. No regular-season games on your date? The notebook prints that clearly rather than using preseason or a different day's games. Training still completes.

A player listed on the roster may be injured or scratched. This version cannot automatically confirm availability. Verify players before using the rankings; use exclusions or `ONLY_PLAYER_IDS` above if needed. Probabilities mean **“if this player appears in this game.”**

In [ ]:
def text_name(value):
    return value.get("default", "") if isinstance(value, dict) else str(value or "")

def slate_candidates(target):
    schedule = get_json(f"{WEB}/schedule/{target:%Y-%m-%d}", ttl_hours=0)
    if "gameWeek" not in schedule:
        raise ValueError("NHL schedule response lacks gameWeek.")
    day_games = [g for day in schedule["gameWeek"] if day.get("date") == str(target.date()) for g in day.get("games", [])]
    regular = [g for g in day_games if g.get("gameType") == 2]
    now = pd.Timestamp.now(tz="UTC")
    games = [g for g in regular if g.get("gameState") in ["FUT", "PRE"]
             and g.get("gameScheduleState", "OK") == "OK"
             and pd.Timestamp(g["startTimeUTC"]) > now]
    print(f"{len(regular)} regular-season games on {target.date()}; {len(games)} eligible games not yet started.")
    if not games:
        print("No eligible games. Change TARGET_DATE and rerun when there is a regular-season slate.")
        return pd.DataFrame(), regular
    rows = []
    for g in games:
        for side, other, home in [("homeTeam", "awayTeam", 1), ("awayTeam", "homeTeam", 0)]:
            team, opponent = g[side]["abbrev"], g[other]["abbrev"]
            if team.upper() in {x.upper() for x in EXCLUDE_TEAMS}:
                continue
            roster = get_json(f"{WEB}/roster/{team}/current", ttl_hours=0)
            if not {"forwards", "defensemen"}.issubset(roster):
                raise ValueError(f"Unexpected roster response for {team}.")
            players = roster["forwards"] + roster["defensemen"]
            if not players:
                raise RuntimeError(f"Empty roster for {team}.")
            for player in players:
                name = (text_name(player["firstName"]) + " " + text_name(player["lastName"])).strip()
                if name.casefold() in {x.casefold() for x in EXCLUDE_PLAYERS}:
                    continue
                if ONLY_PLAYER_IDS and int(player["id"]) not in ONLY_PLAYER_IDS:
                    continue
                rows.append({"game_id": int(g["id"]), "date": target, "start_utc": g["startTimeUTC"],
                             "player_id": int(player["id"]), "player": name, "team": team, "opponent": opponent,
                             "home": home, "position": player.get("positionCode", "D" if player in roster["defensemen"] else "W")})
    future = pd.DataFrame(rows)
    if len(future) and future.duplicated(["game_id", "player_id"]).any():
        raise ValueError("Duplicate slate player-game. Check roster data.")
    return future, regular

future, slate_games = slate_candidates(TARGET)

## 8. Rank players and save predictions
CSV output includes the raw probability (0–1), a display percentage, expected points, fair decimal odds (1/probability), history size, recent role statistics, and availability flags. Fair odds are break-even model estimates before uncertainty or bookmaker margin; they are not offered prices.

Saved snapshots are never overwritten. Every run records the model fingerprint and the time the prediction was made. **Early-roster rankings are not confirmed-lineup rankings.** Players with little history or long absences receive extra flags.

In [ ]:
rankings = pd.DataFrame()
if len(future):
    combined = build_features(history, future)
    candidates = combined[combined.forecast].copy()
    p, expected = predict_bundle(live_bundle, candidates)
    cols = ["date", "start_utc", "game_id", "player_id", "player", "team", "opponent", "home", "position", "prior_games",
            "days_since_played", "points_last20", "shots_last20", "toi_last5", "pp_toi_last5"]
    rankings = candidates[cols].copy()
    rankings["probability"] = p
    rankings["point_chance_pct"] = 100 * p
    rankings["expected_points"] = expected
    rankings["fair_decimal_odds"] = 1 / p
    rankings["availability"] = "ROSTER ONLY — playing status unconfirmed"
    rankings["history_flag"] = np.where(rankings.prior_games < MIN_PRIOR_GAMES, "Limited NHL history", "")
    rankings["layoff_flag"] = np.where(rankings.days_since_played.fillna(999) > 21, "Long layoff / offseason / no history", "")
    rankings["data_through"] = str(history.date.max().date())
    rankings["model_version"] = MODEL_VERSION
    rankings["model_fingerprint"] = fingerprint
    rankings["generated_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
    rankings = rankings.sort_values("probability", ascending=False).reset_index(drop=True)
    rankings.insert(0, "rank", np.arange(1, len(rankings) + 1))
    # Never save a live game as a pregame forecast, even if training/download took a long time.
    rankings = rankings[pd.to_datetime(rankings.start_utc, utc=True) > pd.Timestamp.now(tz="UTC")].copy()
    rankings["rank"] = np.arange(1, len(rankings) + 1)
    if len(rankings):
        stamp = datetime.now(ZoneInfo("UTC")).strftime("%Y%m%dT%H%M%S%fZ")
        output_path = ROOT / "predictions" / f"NHL_Points_{TARGET:%Y-%m-%d}_{stamp}_{fingerprint}.csv"
        rankings.to_csv(output_path, index=False)
        show = ["rank", "player", "team", "opponent", "point_chance_pct", "expected_points", "fair_decimal_odds", "toi_last5", "pp_toi_last5", "history_flag"]
        display(rankings[show].head(TOP_N).round(2))
        print("Saved:", output_path)
        print("All rows are roster candidates, not confirmed starters. Probabilities are conditional on playing.")
    else:
        print("All games started during this run; no pregame snapshot saved. Choose the next slate.")

## 9. Grade previous saved predictions
This section grades prior dates after final results become available. It uses the **first saved pregame prediction per player-game**, so reruns do not inflate the sample. Players absent from the final box score are marked DNP and excluded from model scores; this is not a statement about any sportsbook's settlement rules.

Saving predictions is record keeping, not learning by itself. When you rerun the notebook after new games, new actual results enter the historical dataset and trigger a fresh model fit. The most recent 45 game dates remain reserved for calibration, so recent results influence calibration immediately and model weights as that window moves.

In [ ]:
def grade_saved_predictions():
    paths = sorted((ROOT / "predictions").glob("NHL_Points_*.csv"))
    if not paths:
        print("No saved forecasts yet."); return pd.DataFrame()
    forecasts = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    forecasts["generated_at_utc"] = pd.to_datetime(forecasts.generated_at_utc.map(pd.Timestamp), utc=True)
    forecasts["start_utc"] = pd.to_datetime(forecasts.start_utc.map(pd.Timestamp), utc=True)
    forecasts = forecasts[(forecasts.generated_at_utc < forecasts.start_utc)
                          & (pd.to_datetime(forecasts.date) < TODAY)]
    forecasts = forecasts.sort_values("generated_at_utc").drop_duplicates(["game_id", "player_id"], keep="first")
    if forecasts.empty:
        print("No prior-date forecasts ready to grade."); return pd.DataFrame()
    outcomes = []
    for game_id in forecasts.game_id.unique():
        try:
            box = get_json(f"{WEB}/gamecenter/{int(game_id)}/boxscore", ttl_hours=18)
            if box.get("gameState") not in ["FINAL", "OFF"]:
                continue
            if "playerByGameStats" not in box:
                raise ValueError("Final box score has no playerByGameStats.")
            actual = {}
            for side in ["homeTeam", "awayTeam"]:
                stats = box["playerByGameStats"].get(side, {})
                if not stats.get("forwards") or not stats.get("defense"):
                    raise ValueError("Final box score has incomplete skater groups.")
                for skater in stats["forwards"] + stats["defense"]:
                    if "toi" not in skater or skater["toi"] is None:
                        raise ValueError("Final box score lacks player ice time.")
                    if minutes(skater["toi"]) > 0:
                        actual[int(skater["playerId"])] = int(skater["goals"]) + int(skater["assists"])
            for player_id in forecasts.loc[forecasts.game_id == game_id, "player_id"]:
                pts = actual.get(int(player_id), np.nan)
                outcomes.append({"game_id": int(game_id), "player_id": int(player_id), "actual_points": pts,
                                 "status": "PLAYED" if pd.notna(pts) else "DNP"})
        except (RuntimeError, ValueError, KeyError) as err:
            print(f"Game {game_id} not graded: {err}")
    if not outcomes:
        print("No finalized, complete results ready yet."); return pd.DataFrame()
    graded = forecasts.merge(pd.DataFrame(outcomes), on=["game_id", "player_id"], how="left", validate="one_to_one")
    graded["status"] = graded.status.fillna("PENDING")
    graded.to_csv(ROOT / "evaluation" / "saved_prediction_results.csv", index=False)
    played = graded[graded.status == "PLAYED"]
    print("Status counts:", graded.status.value_counts().to_dict())
    if len(played):
        y = (played.actual_points >= 1).astype(int)
        display(pd.DataFrame([metric_row("Saved pregame forecasts", y, played.probability)]).round(4))
        print("Forecasts graded:", len(played), "| Average probability:", round(played.probability.mean(), 3),
              "| Actual point rate:", round(y.mean(), 3))
        print("Different snapshots may use different model versions. This is a tracking report, not a profit report.")
    return graded

graded_predictions = grade_saved_predictions()

## 10. Optional: download today's CSV to your computer
Set `DOWNLOAD_CSV=True` to request a browser download. With Google Drive enabled, the CSV is already saved in `MyDrive/Nicolay_NHL_Points_Model/predictions`.

In [ ]:
DOWNLOAD_CSV = True
if DOWNLOAD_CSV and len(rankings) and IN_COLAB:
    from google.colab import files
    files.download(str(output_path))

## How to use this each day
- Change the date if needed, then **Run all**. Keep Drive on and the refresh switches off.
- Historical downloads and unchanged models are reused. New games cause an automatic retrain; this is expected.
- To remove a player, put the full name in `EXCLUDE_PLAYERS` and rerun. For slate-only changes in the same live session, rerun Settings and sections 7–10. **If you change TARGET_DATE, rerun everything** so the parsed date updates.
- Compare test Brier/log loss against the simple benchmarks before trusting the rankings. Calibration bins with few observations are noisy. Shared-game player outcomes are correlated, so do not multiply these probabilities to price a parlay.
- Early in a season, prior-season form dominates. Trades, new roles, injuries, and rookies can make the historical estimate less reliable.
- The first real-data run may expose changes to NHL endpoints. If an error appears, send the full traceback; do not replace failed downloads with zeros.

### Data and implementation references
- NHL official statistics service: https://api.nhle.com/stats/rest/en/
- NHL official schedule and roster service: https://api-web.nhle.com/v1/
- Community endpoint reference (not an NHL support guarantee): https://github.com/Zmalski/NHL-API-Reference
- Scikit-learn gradient boosting: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html

This notebook does not include preseason/playoff forecasts, live injury checks, starting-goalie adjustments, expected-goals data, automatic bet placement. Section 11 supports an automatic odds feed after one-time API-key setup, with manual entry as an optional fallback. Those are possible later additions after the base model has real out-of-sample results.

## 11. Automatic odds — daily budget mode
Create `ODDS_API_KEY` in Colab Secrets and enable notebook access. Reuse the same key in both notebooks; never paste it into code. Both share your account's 500 monthly credits.

**Normal daily run:** leave `ODDS_REFRESH_TOKEN=""` in Settings. Fetches only unstarted games in the selected slate, one player-points market, and four specified books (up to one credit per game). Results—including empty markets—are saved in Drive and reused for the rest of the New York calendar day. New days fetch again. Rerunning or retraining does not refresh odds automatically. Keep Google Drive enabled for reuse across sessions.

**Deliberate refresh:** change `ODDS_REFRESH_TOKEN` to a new label (for example `pregame-1`), rerun Settings, then the two odds cells below. Each new label allows one refresh per game that day; leaving the label unchanged does not repeatedly spend credits. Clear it afterward. Newly added games can still cost credits. Use a different label for another intentional refresh.

Only standard `player_points` is fetched. If the feed lacks 0.5 lines, it reports missing coverage and does not spend more trying other markets. You may replace the setting with `["player_points_alternate"]` to try that single market deliberately; changing markets costs another fetch. No automatic fallback.

A free quota check displays the live account balance. New odds requests stop at the 25-credit reserve or 20-credit run limit; predictions still work. The reserve protects the account total, not separate NFL/NHL monthly allocations. These guards cannot promise a full month's coverage if schedules or refresh frequency exceed your budget.

**Saved does not mean current:** original quote timestamps remain intact. A saved-price table stays visible; quotes older than one hour are excluded from the current value board. Update the refresh token to get current prices. Underdog candidates also show quote age and freshness; indicative DFS prices are never treated as executable straight-bet odds. Check actual entry payouts and settlement rules.

Sportsbook EV = model probability × decimal odds − 1. The 3-percentage-point probability reduction is a sensitivity check, not a confidence interval. Missing under prices mean no no-vig estimate. No historical odds profitability test is claimed. Manual odds input remains available via `ODDS_SOURCE="manual"`.

Provider references: [API and quota rules](https://the-odds-api.com/liveapi/guides/v4/), [markets](https://the-odds-api.com/sports-odds-data/betting-markets.html). Offline tests cover caching, quota stops, timestamps, identity matching and parsing; authenticated live fetching requires your key.


In [ ]:
ODDS_SOURCE = "api"               # "api" = automatic; "manual" = optional fallback.
# Enter real quotes here. The commented example is only a format guide, not a real price.
ODDS_INPUT = [
    # {"player": "PLAYER NAME FROM RANKINGS", "book": "YOUR BOOK", "line": 0.5,
    #  "over_american": 130, "under_american": -150,
    #  "quoted_at_utc": "YYYY-MM-DDTHH:MM:SS-04:00", "playing_confirmed": False},
]
ODDS_CSV = ""                     # Optional path; takes precedence over ODDS_INPUT.
MAX_ODDS_AGE_HOURS = 1.0          # Recheck older prices before using them.
MIN_ESTIMATED_EV_PCT = 3.0        # Review threshold, not a guarantee of value.
PROBABILITY_BUFFER_PP = 3.0       # Lower model probability by 3 percentage points as a stress test.
MAX_PREDICTION_AGE_HOURS = 24.0

# A blank template is saved only when this slate has not already received one.
odds_folder = ROOT / "odds"
odds_folder.mkdir(exist_ok=True)
if len(rankings) and ODDS_SOURCE == "manual":
    odds_template_path = odds_folder / f"NHL_Odds_Input_{TARGET:%Y-%m-%d}.csv"
    if not odds_template_path.exists():
        template = rankings[["game_id", "player_id", "player", "team", "opponent"]].copy()
        template["book"] = ""
        template["line"] = 0.5
        template["over_american"] = ""
        template["under_american"] = ""
        template["quoted_at_utc"] = ""
        template["playing_confirmed"] = False
        template.to_csv(odds_template_path, index=False)
    print("Optional odds-entry template:", odds_template_path)

def american_to_decimal(odds):
    # American odds express net winnings relative to a 100-unit stake or payout.
    value = float(str(odds).strip().replace("−", "-"))
    if not np.isfinite(value) or abs(value) < 100:
        raise ValueError("American odds must be >= +100 or <= -100. Use +100 for even money.")
    return 1 + (value / 100 if value > 0 else 100 / abs(value))

def fair_american(probability):
    if probability >= 0.5:
        return -100 * probability / (1 - probability)
    return 100 * (1 - probability) / probability

def has_value(value):
    return value is not None and not pd.isna(value) and str(value).strip() != ""

def explicit_utc(value):
    if not has_value(value):
        raise ValueError("Missing odds timestamp.")
    result = pd.Timestamp(value)
    if pd.isna(result) or result.tzinfo is None:
        raise ValueError("Timestamp must include a timezone, e.g. Z or -04:00.")
    return result.tz_convert("UTC")

def confirmed_bool(value):
    # Do not treat the CSV string 'False' as True.
    return str(value).strip().casefold() in {"true", "yes", "1"}

def compare_odds(ranked, quotes, now=None):
    now = pd.Timestamp.now(tz="UTC") if now is None else explicit_utc(now)
    if not 0 <= PROBABILITY_BUFFER_PP < 100:
        raise ValueError("Probability buffer must be between 0 and 100 percentage points.")
    if MAX_ODDS_AGE_HOURS <= 0 or MAX_PREDICTION_AGE_HOURS <= 0:
        raise ValueError("Freshness limits must be positive.")
    accepted, rejected = [], []
    for row_number, row in enumerate(quotes.to_dict("records"), start=1):
        if not has_value(row.get("over_american")):
            continue                  # Unfilled template rows are not errors.
        try:
            if float(row.get("line", 0.5)) != 0.5:
                raise ValueError("Only the over 0.5 points market matches this model.")
            book = str(row.get("book", "")).strip() if has_value(row.get("book")) else ""
            if not book:
                raise ValueError("Missing sportsbook name.")
            if book.casefold().replace(" ", "") in {"underdog", "underdogfantasy", "underdogsports"}:
                raise ValueError("Underdog DFS prices are indicative; use the separate Underdog candidate table.")
            matches = ranked.copy()
            if has_value(row.get("player_id")):
                number = float(row["player_id"])
                if not number.is_integer():
                    raise ValueError("Invalid player ID.")
                matches = matches[matches.player_id == int(number)]
            elif has_value(row.get("player")):
                matches = matches[matches.player.str.strip().str.casefold() == str(row["player"]).strip().casefold()]
            else:
                raise ValueError("Provide player_id or an exact player name.")
            if has_value(row.get("game_id")):
                number = float(row["game_id"])
                if not number.is_integer():
                    raise ValueError("Invalid game ID.")
                matches = matches[matches.game_id == int(number)]
            if len(matches) != 1:
                raise ValueError("Player/game does not match exactly one current ranking. Check name and IDs.")
            player = matches.iloc[0]
            # Guard against a copied price row whose name disagrees with its ID.
            if has_value(row.get("player")) and str(row["player"]).strip().casefold() != str(player.player).strip().casefold():
                raise ValueError("Player name does not match the supplied ID.")
            if has_value(row.get("team")) and str(row["team"]).upper().strip() != str(player.team).upper():
                raise ValueError("Team does not match the current ranking.")
            quoted = explicit_utc(row.get("quoted_at_utc"))
            quote_age = (now - quoted).total_seconds() / 3600
            if quote_age < -5 / 60:
                raise ValueError("Quote timestamp is in the future.")
            if quote_age > MAX_ODDS_AGE_HOURS:
                raise ValueError("Odds are stale; recheck the price and timestamp.")
            if explicit_utc(player.start_utc) <= now:
                raise ValueError("Game has already started.")
            prediction_age = (now - explicit_utc(player.generated_at_utc)).total_seconds() / 3600
            if prediction_age > MAX_PREDICTION_AGE_HOURS or prediction_age < -5 / 60:
                raise ValueError("Prediction timestamp is stale or invalid; rerun rankings.")
            probability = float(player.probability)
            if not np.isfinite(probability) or not 0 < probability < 1:
                raise ValueError("Invalid model probability.")
            decimal = american_to_decimal(row["over_american"])
            implied = 1 / decimal
            market_probability, overround = np.nan, np.nan
            under = row.get("under_american")
            if has_value(under):
                under_implied = 1 / american_to_decimal(under)
                market_probability = implied / (implied + under_implied)
                overround = 100 * (implied + under_implied - 1)
            buffered_probability = max(0, probability - PROBABILITY_BUFFER_PP / 100)
            ev_pct = 100 * (probability * decimal - 1)
            buffered_ev = 100 * (buffered_probability * decimal - 1)
            enough_history = int(player.prior_games) >= MIN_PRIOR_GAMES
            confirmed = confirmed_bool(row.get("playing_confirmed", False))
            flags = []
            if not confirmed:
                flags.append("Playing status unconfirmed")
            if not enough_history:
                flags.append("Limited NHL history")
            if has_value(player.get("layoff_flag")):
                flags.append(str(player.layoff_flag))
            if pd.notna(overround) and overround < 0:
                flags.append("Over/under pair implies negative margin; verify prices")
            accepted.append({
                "player": player.player, "player_id": int(player.player_id), "game_id": int(player.game_id),
                "team": player.team, "opponent": player.opponent, "book": book, "line": 0.5,
                "over_american": float(str(row["over_american"]).replace("−", "-")),
                "under_american": float(str(under).replace("−", "-")) if has_value(under) else np.nan,
                "decimal_odds": decimal, "model_chance_pct": 100 * probability,
                "break_even_pct": 100 * implied, "edge_pp": 100 * (probability - implied),
                "estimated_ev_pct": ev_pct, "estimated_net_per_100_staked": ev_pct,
                "buffered_ev_pct": buffered_ev, "fair_american": fair_american(probability),
                "market_no_vig_pct": 100 * market_probability,
                "no_vig_edge_pp": 100 * (probability - market_probability), "book_overround_pct": overround,
                "quote_age_minutes": max(0, quote_age * 60), "quoted_at_utc": quoted.isoformat(),
                "playing_confirmed": confirmed, "flags": "; ".join(flags),
                "passes_value_filters": bool(ev_pct >= MIN_ESTIMATED_EV_PCT and buffered_ev > 0 and enough_history),
                "prediction_generated_at_utc": str(player.generated_at_utc),
                "model_fingerprint": player.model_fingerprint,
            })
        except (ValueError, TypeError, KeyError, OverflowError) as err:
            rejected.append({"input_row": row_number, "player": row.get("player", row.get("player_id", "")),
                             "book": row.get("book", ""), "reason": str(err)})
    board = pd.DataFrame(accepted)
    if len(board):
        # Multiple versions from one book: use its latest quote, not the most attractive old one.
        board = board.sort_values("quoted_at_utc").drop_duplicates(["game_id", "player_id", "book"], keep="last")
        board = board.sort_values("estimated_ev_pct", ascending=False).reset_index(drop=True)
        board.insert(0, "value_rank", np.arange(1, len(board) + 1))
    return board, pd.DataFrame(rejected)

In [ ]:
# AUTO_ODDS_CELL: rerunning reuses today's saved prices. Change ODDS_REFRESH_TOKEN in Settings for a deliberate refresh.
import unicodedata, re
ODDS_BOOKMAKERS = ["fanduel", "draftkings", "betmgm", "underdog"]
ODDS_MARKETS = ["player_points"]  # ONE market. To try alternates, replace with ["player_points_alternate"].
PLAYER_NAME_ALIASES = {}  # Exact provider-name -> exact model-name only.

# Shared behavior in both notebooks; no API key is written to disk.
import os, json, hashlib, math, sys, time, urllib.request, urllib.parse, urllib.error
from pathlib import Path
import pandas as pd

class OddsFeedError(RuntimeError):
    pass

class BudgetOddsClient:
    def __init__(self, root, api_key, refresh_token='', reserve=25, run_limit=20):
        self.api_key = api_key
        self.root = Path(root) / 'odds_daily_cache'
        self.root.mkdir(parents=True, exist_ok=True)
        self.refresh_token = str(refresh_token).strip()
        self.reserve, self.run_limit = int(reserve), int(run_limit)
        if self.reserve < 0 or self.run_limit < 0:
            raise OddsFeedError('Credit limits cannot be negative.')
        self.remaining = None
        self.spent = 0
        self.account = hashlib.sha256(api_key.encode()).hexdigest()[:16]

    def today(self):
        return str(pd.Timestamp.now(tz='America/New_York').date())

    def _network(self, path, params):
        # No automatic retries: an interrupted response might already have cost credits.
        query = urllib.parse.urlencode(dict(params, apiKey=self.api_key))
        req = urllib.request.Request('https://api.the-odds-api.com/v4/' + path + '?' + query,
                                     headers={'Accept':'application/json'})
        try:
            with urllib.request.urlopen(req, timeout=35) as response:
                raw = response.read()
                headers = dict(response.headers)
            data = json.loads(raw)
        except urllib.error.HTTPError as err:
            raise OddsFeedError(f'Odds request stopped (HTTP {err.code}). Check key, coverage or quota; no automatic retry.') from None
        except (urllib.error.URLError, TimeoutError, OSError, ValueError):
            raise OddsFeedError('Odds response unavailable. No automatic retry; check connection and account usage before retrying.') from None
        if not isinstance(data, (dict,list)) or (isinstance(data,dict) and 'error_code' in data):
            raise OddsFeedError('Unexpected odds response; check the provider dashboard.')
        return data, {str(k).lower():v for k,v in headers.items()}

    def check_quota(self):
        # /sports is free; live quota covers both notebooks and other uses of this key.
        _, headers = self._network('sports', {})
        try:
            self.remaining = int(float(headers['x-requests-remaining']))
        except (KeyError, TypeError, ValueError):
            raise OddsFeedError('Could not verify remaining credits; paid requests disabled.') from None
        print(f'Account credits remaining: {self.remaining} | Protected reserve: {self.reserve}')

    def request(self, path, params):
        chargeable = path.endswith('/odds')
        if not chargeable:
            data, _ = self._network(path, params)  # Event discovery is free and stays current.
            return data
        markets = set(filter(None, str(params.get('markets','')).split(',')))
        books = set(filter(None, str(params.get('bookmakers','')).split(',')))
        if len(markets) != 1 or not 1 <= len(books) <= 10:
            raise OddsFeedError('Budget mode requires exactly one market and 1–10 explicit bookmakers.')
        cache_key = hashlib.sha256(json.dumps([self.account,path,params],sort_keys=True).encode()).hexdigest()
        cache_path = self.root / (cache_key + '.json')
        cached = None
        if cache_path.exists():
            try:
                cached = json.loads(cache_path.read_text())
            except (ValueError,OSError):
                raise OddsFeedError('Saved odds cache is unreadable. Inspect it before spending another request.') from None
            # An unchanged token never repeatedly refreshes, even across Colab restarts.
            if cached.get('fetch_day_et') == self.today() and (not self.refresh_token or cached.get('refresh_token') == self.refresh_token):
                print('SAVED ODDS:', cached.get('fetched_at_utc'), '| 0 credits; quote times remain unchanged.')
                return cached['payload']
        if self.remaining is None:
            self.check_quota()
        if self.remaining - 1 < self.reserve or self.spent + 1 > self.run_limit:
            raise OddsFeedError('Credit guard stopped new odds requests. Saved results remain available; predictions are unaffected.')
        # Reserve one credit before attempting, even if the response is interrupted.
        self.spent += 1
        self.remaining -= 1
        data, headers = self._network(path, params)
        try:
            self.remaining = int(float(headers['x-requests-remaining']))
        except (KeyError, TypeError, ValueError):
            self.remaining = None  # Force a free authoritative quota check before any next fetch.
        print(f'New odds fetch: cost {headers.get("x-requests-last", "up to 1")}; account remaining: {self.remaining}')
        record = {'fetch_day_et':self.today(), 'fetched_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),
                  'refresh_token':self.refresh_token, 'payload':data}
        tmp = cache_path.with_suffix('.tmp')
        tmp.write_text(json.dumps(record)); tmp.replace(cache_path)
        return data

def odds_api_key():
    key = os.environ.get('ODDS_API_KEY','').strip()
    if key:
        return key
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            return str(userdata.get('ODDS_API_KEY')).strip()
        except Exception:
            pass
    return ''

def odds_request(path, params, api_key):
    return _odds_client.request(path, params)

def normalize_label(value):
    plain = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]", "", plain)

# Explicit aliases only: unknown teams are reported, never assigned by fuzzy matching.
TEAM_NAMES = {
    "Anaheim Ducks":"ANA", "Boston Bruins":"BOS", "Buffalo Sabres":"BUF", "Calgary Flames":"CGY",
    "Carolina Hurricanes":"CAR", "Chicago Blackhawks":"CHI", "Colorado Avalanche":"COL",
    "Columbus Blue Jackets":"CBJ", "Dallas Stars":"DAL", "Detroit Red Wings":"DET",
    "Edmonton Oilers":"EDM", "Florida Panthers":"FLA", "Los Angeles Kings":"LAK",
    "Minnesota Wild":"MIN", "Montreal Canadiens":"MTL", "Nashville Predators":"NSH",
    "New Jersey Devils":"NJD", "New York Islanders":"NYI", "New York Rangers":"NYR",
    "Ottawa Senators":"OTT", "Philadelphia Flyers":"PHI", "Pittsburgh Penguins":"PIT",
    "San Jose Sharks":"SJS", "Seattle Kraken":"SEA", "St Louis Blues":"STL",
    "Tampa Bay Lightning":"TBL", "Toronto Maple Leafs":"TOR", "Utah Mammoth":"UTA",
    "Utah Hockey Club":"UTA", "Vancouver Canucks":"VAN", "Vegas Golden Knights":"VGK",
    "Washington Capitals":"WSH", "Winnipeg Jets":"WPG", "Arizona Coyotes":"ARI",
}
TEAM_ALIASES = {normalize_label(k):v for k,v in TEAM_NAMES.items()}
TEAM_ALIASES.update({normalize_label(v):v for v in TEAM_NAMES.values()})

def match_event(event, ranked):
    home = TEAM_ALIASES.get(normalize_label(event.get("home_team", "")))
    away = TEAM_ALIASES.get(normalize_label(event.get("away_team", "")))
    if not home or not away:
        return None
    matched = []
    for gid, group in ranked.groupby("game_id"):
        row = group.iloc[0]
        nhl_home, nhl_away = (row.team, row.opponent) if row.home else (row.opponent, row.team)
        if (home, away) != (nhl_home, nhl_away):
            continue
        # Team pair AND start time must agree; a different day's game cannot attach to this slate.
        if abs((explicit_utc(event["commence_time"]) - explicit_utc(row.start_utc)).total_seconds()) <= 3600:
            matched.append(int(gid))
    return matched[0] if len(matched) == 1 else None

def parse_event_odds(event, game_id, ranked, now=None):
    now = pd.Timestamp.now(tz="UTC") if now is None else explicit_utc(now)
    sportsbooks, underdog, issues = [], [], []
    candidates = ranked[ranked.game_id == game_id].copy()
    candidates["normalized_name"] = candidates.player.map(normalize_label)
    if match_event(event, candidates) != game_id:
        return [], [], [{"event":event.get("id"), "reason":"Event identity changed; skipped"}]
    for book in event.get("bookmakers", []):
        bkey = book.get("key")
        if bkey not in ODDS_BOOKMAKERS:
            continue
        for market in book.get("markets", []):
            if market.get("key") not in ODDS_MARKETS:
                continue
            try:
                updated = explicit_utc(market.get("last_update"))
                age = (now - updated).total_seconds() / 3600
                if age < -5/60:
                    raise ValueError("Future market timestamp")
            except (ValueError, TypeError) as err:
                issues.append({"book":bkey, "reason":str(err)}); continue
            outcomes = market.get("outcomes", [])
            for outcome in outcomes:
                try:
                    if str(outcome.get("name", "")).casefold() not in {"over", "higher"} or float(outcome.get("point", -1)) != .5:
                        continue
                    source_name = str(outcome.get("description", ""))
                    model_name = PLAYER_NAME_ALIASES.get(source_name, source_name)
                    matches = candidates[candidates.normalized_name == normalize_label(model_name)]
                    if len(matches) != 1:
                        raise ValueError("Player name missing/ambiguous in this game's rankings")
                    player = matches.iloc[0]
                    if explicit_utc(player.start_utc) <= now:
                        continue
                    row = {"game_id":game_id, "player_id":int(player.player_id), "player":player.player,
                           "team":player.team, "book":book.get("title", bkey), "line":.5,
                           "quoted_at_utc":updated.isoformat(), "playing_confirmed":False,
                           "source_event_id":event.get("id"), "source_market":market["key"]}
                    if bkey == "underdog":
                        row.update(model_chance_pct=100*float(player.probability),
                                   reported_multiplier=outcome.get("multiplier", np.nan),
                                   availability="Provider-listed selection; playing status unconfirmed",
                                   prior_games=int(player.prior_games),
                                   note="Indicative DFS feed. Actual entry payout required for Underdog EV.")
                        underdog.append(row)   # No indicative price passed to compare_odds.
                    else:
                        american_to_decimal(outcome["price"])  # Validate before accepting.
                        row["over_american"] = outcome["price"]
                        pairs = [o for o in outcomes if str(o.get("name", "")).casefold()=="under"
                                 and o.get("description")==outcome.get("description") and o.get("point")==outcome.get("point")]
                        row["under_american"] = pairs[0].get("price", np.nan) if len(pairs)==1 else np.nan
                        sportsbooks.append(row)
                except (ValueError, TypeError, KeyError) as err:
                    issues.append({"book":bkey,"player":outcome.get("description", ""),"reason":str(err)})
    return sportsbooks, underdog, issues

def fetch_slate_odds(ranked, api_key):
    ranked = ranked[pd.to_datetime(ranked.start_utc, utc=True) > pd.Timestamp.now(tz="UTC")].copy()
    if ranked.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    events = odds_request("sports/icehockey_nhl/events", {"dateFormat":"iso"}, api_key)
    if not isinstance(events, list):
        raise OddsFeedError("Unexpected events response.")
    mapping, issues = {}, []
    for event in events:
        try:
            gid = match_event(event, ranked)
        except (ValueError, KeyError, TypeError):
            continue
        if gid is not None:
            mapping.setdefault(gid, []).append(event)
    sportsbook_rows, underdog_rows = [], []
    for game_id in ranked.game_id.unique():
        possible = mapping.get(int(game_id), [])
        if len(possible) != 1:
            issues.append({"game_id":int(game_id),"reason":"No unique odds-provider event matched this NHL game"})
            continue
        event = possible[0]
        try:
            event_id = str(event["id"])
            if not re.fullmatch(r"[A-Za-z0-9_-]+", event_id):
                raise OddsFeedError("Invalid provider event ID.")
            response = odds_request(f"sports/icehockey_nhl/events/{event_id}/odds",
                                     {"bookmakers":",".join(ODDS_BOOKMAKERS), "markets":",".join(ODDS_MARKETS),
                                      "oddsFormat":"american", "dateFormat":"iso", "includeMultipliers":"true"}, api_key)
            a,b,c = parse_event_odds(response, int(game_id), ranked)
            sportsbook_rows.extend(a);underdog_rows.extend(b);issues.extend(c)
        except OddsFeedError as err:
            # Key/quota/network failure stops this fetch rather than repeatedly spending requests.
            print("Automatic odds stopped:", str(err));break
    matched_ids = set(mapping)
    print(f"Odds events matched: {len(matched_ids)}/{ranked.game_id.nunique()}; "
          f"sportsbook quotes: {len(sportsbook_rows)}; Underdog selections: {len(underdog_rows)}")
    return pd.DataFrame(sportsbook_rows), pd.DataFrame(underdog_rows), pd.DataFrame(issues)

auto_quotes, underdog_candidates, odds_feed_issues = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if ODDS_SOURCE == "api" and FETCH_ODDS and len(rankings):
    _key = odds_api_key()
    if not _key:
        print("ONE-TIME SETUP: create ODDS_API_KEY in Colab Secrets and enable notebook access; then rerun this cell.")
    else:
        try:
            _odds_client = BudgetOddsClient(ROOT, _key, ODDS_REFRESH_TOKEN, ODDS_CREDIT_RESERVE, ODDS_MAX_CREDITS_PER_RUN)
            _odds_client.check_quota()
            auto_quotes, underdog_candidates, odds_feed_issues = fetch_slate_odds(rankings, _key)
            print(f'New paid fetch attempts this run: {_odds_client.spent}')
            if len(odds_feed_issues):
                print("Missing or excluded feed data:");display(odds_feed_issues)
        except OddsFeedError as err:
            print(str(err))
        finally:
            del _key
elif ODDS_SOURCE not in {"api", "manual"}:
    raise ValueError("ODDS_SOURCE must be api or manual.")

In [ ]:
if len(auto_quotes):
    saved_prices = auto_quotes.copy()
    saved_prices["quote_age_minutes"] = (pd.Timestamp.now(tz="UTC") - pd.to_datetime(saved_prices.quoted_at_utc, utc=True)).dt.total_seconds() / 60
    saved_prices["price_status"] = np.where(saved_prices.quote_age_minutes <= MAX_ODDS_AGE_HOURS*60, "Recent quote — verify before use", "SAVED / STALE — refresh before use")
    print("SAVED PRICE SNAPSHOT — may differ from current offers:")
    display(saved_prices[["player","book","over_american","quoted_at_utc","quote_age_minutes","price_status"]].head(TOP_N))

value_board, rejected_odds = pd.DataFrame(), pd.DataFrame()
if len(rankings):
    quotes = auto_quotes.copy() if ODDS_SOURCE == "api" else (pd.read_csv(ODDS_CSV) if ODDS_CSV else pd.DataFrame(ODDS_INPUT))
    if quotes.empty:
        print("No usable sportsbook odds available. Check API setup, coverage, and whether props have been posted." if ODDS_SOURCE == "api" else "No manual odds entered.")
    else:
        value_board, rejected_odds = compare_odds(rankings, quotes)
        if len(rejected_odds):
            print("Quotes excluded from the value board:")
            display(rejected_odds)
        if len(value_board):
            columns = ["player", "team", "book", "over_american", "model_chance_pct", "break_even_pct",
                       "edge_pp", "estimated_ev_pct", "buffered_ev_pct", "flags"]
            print("VALUE BOARD — highest estimated return first (all usable submitted prices):")
            display(value_board[columns].head(TOP_N).round(2))
            best_prices = value_board.drop_duplicates(["game_id", "player_id"]).copy()
            print("BEST ENTERED PRICE PER PLAYER:")
            display(best_prices[columns].head(TOP_N).round(2))
            review_candidates = best_prices[best_prices.passes_value_filters]
            print("REVIEW CANDIDATES — positive after the chosen buffer; check playing status and market rules:")
            display(review_candidates[columns].head(TOP_N).round(2))
            if review_candidates.empty:
                print("None pass the current value filters. No need to force a selection.")
            if model_brier >= baseline_brier:
                print("MODEL CHECK: this model did not beat the recent-player baseline on test Brier score. These edges are experimental.")
            stamp = datetime.now(ZoneInfo("UTC")).strftime("%Y%m%dT%H%M%S%fZ")
            value_path = odds_folder / f"NHL_Value_{TARGET:%Y-%m-%d}_{stamp}.csv"
            value_board["compared_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
            value_board.to_csv(value_path, index=False)
            if len(rejected_odds):
                rejected_odds.to_csv(value_path.with_name(value_path.stem + "_excluded.csv"), index=False)
            print("Saved value snapshot:", value_path)
            print("EV assumes a 1-unit straight bet and the model probability is correct, conditional on playing.")
        else:
            print("No usable quotes. Check the excluded rows above.")
else:
    print("No current rankings. Run the regular-season slate before comparing prices.")

if len(underdog_candidates):
    ud = underdog_candidates.copy()
    ud["quote_age_minutes"] = (pd.Timestamp.now(tz="UTC")-pd.to_datetime(ud.quoted_at_utc,utc=True)).dt.total_seconds()/60
    ud["price_status"] = np.where(ud.quote_age_minutes <= MAX_ODDS_AGE_HOURS*60, "Recent — verify in app", "SAVED / STALE — refresh")
    # Keep variants with distinct multipliers separate; none are assigned a single-bet EV.
    ud = ud.sort_values("quoted_at_utc").drop_duplicates(
        ["game_id", "player_id", "source_market", "reported_multiplier"], keep="last")
    if len(value_board):
        market = value_board.dropna(subset=["market_no_vig_pct"]).groupby(["game_id", "player_id"], as_index=False).agg(
            sportsbook_consensus_pct=("market_no_vig_pct", "median"), books_in_consensus=("book", "nunique"))
        ud = ud.merge(market, on=["game_id", "player_id"], how="left", validate="many_to_one")
    else:
        ud["sportsbook_consensus_pct"] = np.nan
        ud["books_in_consensus"] = 0
    ud["model_minus_market_pp"] = ud.model_chance_pct - ud.sportsbook_consensus_pct
    ud = ud.sort_values(["model_minus_market_pp", "model_chance_pct"], ascending=False, na_position="last")
    print("UNDERDOG CANDIDATES — model vs sportsbook probability, NOT Underdog expected return:")
    display(ud[["player", "team", "line", "reported_multiplier", "model_chance_pct", "sportsbook_consensus_pct",
                "model_minus_market_pp", "books_in_consensus", "quote_age_minutes", "price_status", "availability"]].head(TOP_N).round(2))
    print("No displayed multiplier means the provider did not supply it. Verify the actual entry payout in Underdog.")
    stamp = datetime.now(ZoneInfo("UTC")).strftime("%Y%m%dT%H%M%S%fZ")
    ud.to_csv(odds_folder / f"NHL_Underdog_Candidates_{TARGET:%Y-%m-%d}_{stamp}.csv", index=False)
elif ODDS_SOURCE == "api" and len(rankings):
    print("No matched Underdog 0.5-point selections returned. Coverage is not guaranteed.")


## 12. Automatic live performance reports
Rates are fractions. Positive actual-minus-predicted values mean more hits than predicted. These reports do not alter the model.


In [ ]:
# Automatic diagnostics: this does not change probabilities or retrain on graded forecasts.
def performance_reports(graded, folder):
    if graded.empty:
        print("No finalized saved predictions yet.")
        return {}
    played = graded.loc[graded.status.eq("PLAYED")].copy()
    if played.empty:
        print("No played appearances to evaluate yet.")
        return {}
    played["hit"] = played.actual_points.ge(1).astype(int)
    played["brier"] = (played.probability - played.hit) ** 2
    played["probability_band"] = pd.cut(played.probability,
        [0, .3, .4, .5, .6, .7, 1.000001], right=False,
        labels=["Below 30%", "30–40%", "40–50%", "50–60%", "60–70%", "70%+"])
    reports = {}
    for name, key in [("by_probability", "probability_band"), ("by_date", "date")]:
        report = played.groupby(key, observed=True).agg(
            appearances=("hit", "size"), games=("game_id", "nunique"),
            expected_hits=("probability", "sum"), actual_hits=("hit", "sum"),
            predicted_rate=("probability", "mean"), actual_rate=("hit", "mean"),
            brier=("brier", "mean")).reset_index()
        report["actual_minus_predicted_pp"] = 100 * (report.actual_rate - report.predicted_rate)
        report.to_csv(folder / f"live_{name}.csv", index=False)
        reports[name] = report
        print(name.replace("_", " ").upper())
        display(report.round(4))
    print(f"{len(played):,} appearances across {played.game_id.nunique()} games. "
          "Daily samples differ; daily scores do not establish model improvement. "
          "Players in the same game are correlated. Small bands are noisy.")
    return reports

live_reports = performance_reports(graded_predictions, ROOT / "evaluation")


## 13. Automatic prospective simulation
No actual wagers are placed or requested. Rules are fixed below; results use recorded pregame selections and available finalized outcomes.


In [ ]:
# Fixed prospective strategy. No backfilled picks and no actual wagers required.
# These are research defaults, not optimized or proven profitable thresholds.
PAPER_RULES = {"version": "paper_v1", "min_ev_pct": 3.0, "buffer_pp": 3.0,
               "min_history": 10, "max_quote_age_minutes": 60.0,
               "max_prediction_age_hours": 24.0, "stake_units": 1.0}

def paper_candidates(board, ranking_rows, now):
    if board.empty or ranking_rows.empty:
        return pd.DataFrame()
    d = board.merge(ranking_rows[["game_id", "player_id", "date", "start_utc",
                                  "prior_games", "layoff_flag"]],
                    on=["game_id", "player_id"], validate="many_to_one")
    quote = pd.to_datetime(d.quoted_at_utc, utc=True, errors="coerce")
    predicted = pd.to_datetime(d.prediction_generated_at_utc, utc=True, errors="coerce")
    start = pd.to_datetime(d.start_utc, utc=True, errors="coerce")
    age = (now - quote).dt.total_seconds() / 60
    pred_age = (now - predicted).dt.total_seconds() / 3600
    p = d.model_chance_pct / 100
    d["paper_ev_pct"] = 100 * (p * d.decimal_odds - 1)
    d["paper_buffered_ev_pct"] = 100 * ((p - PAPER_RULES["buffer_pp"] / 100).clip(lower=0) * d.decimal_odds - 1)
    eligible = (start.gt(now) & age.between(0, PAPER_RULES["max_quote_age_minutes"])
        & pred_age.between(0, PAPER_RULES["max_prediction_age_hours"])
        & d.prior_games.ge(PAPER_RULES["min_history"])
        & d.layoff_flag.fillna("").eq("")
        & d.paper_ev_pct.ge(PAPER_RULES["min_ev_pct"])
        & d.paper_buffered_ev_pct.gt(0)
        & d.decimal_odds.gt(1) & np.isfinite(d.decimal_odds)
        & ~d.book.str.lower().str.contains("underdog", na=False))
    # One hypothetical straight per player-game, at the best qualifying current book.
    d = d.loc[eligible].sort_values(["decimal_odds", "book"], ascending=[False, True])
    d = d.drop_duplicates(["game_id", "player_id"]).copy()
    d["stake_units"] = PAPER_RULES["stake_units"]
    d["selected_at_utc"] = now.isoformat()
    return d

def settle_paper(ledger, graded):
    if ledger.empty:
        return ledger.copy()
    if graded.empty:
        result = ledger.copy()
        result["status"] = "PENDING"
        result["actual_points"] = np.nan
    else:
        outcomes = graded[["game_id", "player_id", "status", "actual_points"]]
        result = ledger.merge(outcomes, on=["game_id", "player_id"], how="left", validate="many_to_one")
        result["status"] = result.status.fillna("PENDING")
    played = result.status.eq("PLAYED")
    result["profit_units"] = np.nan
    result.loc[played, "profit_units"] = np.where(
        result.loc[played, "actual_points"].ge(1),
        result.loc[played, "stake_units"] * (result.loc[played, "decimal_odds"] - 1),
        -result.loc[played, "stake_units"])
    result.loc[result.status.eq("DNP"), "profit_units"] = 0.0
    return result

def run_paper_tracking(board, ranking_rows, graded):
    rule_id = hashlib.sha256(json.dumps(PAPER_RULES, sort_keys=True).encode()).hexdigest()[:12]
    folder = ROOT / "evaluation" / "paper" / rule_id
    folder.mkdir(parents=True, exist_ok=True)
    atomic_json(folder / "rules.json", PAPER_RULES)
    snapshot = folder / f"selections_{TARGET:%Y-%m-%d}.json"
    now = pd.Timestamp.now(tz="UTC")
    # Missing odds do not freeze an empty day; a later successful run can initialize it.
    if not snapshot.exists() and not board.empty and not ranking_rows.empty:
        picks = paper_candidates(board, ranking_rows, now)
        payload = {"selected_at_utc": now.isoformat(), "date": str(TARGET.date()),
                   "rules_id": rule_id, "picks": json.loads(picks.to_json(orient="records", date_format="iso"))}
        atomic_json(snapshot, payload)
        print(f"Saved {len(picks)} simulated selections. This date is now frozen for these rules.")
    elif snapshot.exists():
        print("Reusing this date's frozen simulated selections; reruns do not add or replace picks.")
    else:
        print("No usable odds/slate: no simulated selections recorded this run.")
    records = []
    for path in sorted(folder.glob("selections_*.json")):
        records.extend(json.loads(path.read_text(encoding="utf-8"))["picks"])
    if not records:
        print("No simulated bets yet. Zero-pick days are allowed.")
        return pd.DataFrame()
    ledger = pd.DataFrame(records)
    if ledger.duplicated(["game_id", "player_id"]).any():
        raise ValueError("Duplicate player-game in simulated ledger; investigate before reporting.")
    result = settle_paper(ledger, graded)
    result.to_csv(folder / "simulated_results.csv", index=False)
    settled = result[result.status.eq("PLAYED")].copy()
    print("SIMULATION ONLY: saved sportsbook quotes, 1 unit per straight, DNP refunded. "
          "Not personal profit, not Underdog returns, and no assumption all quotes were accessible. "
          "Playing status is not confirmed. No historical selections are invented.")
    print("Status counts:", result.status.value_counts().to_dict())
    if len(settled):
        daily = settled.groupby("date").agg(bets=("player_id", "size"),
                    stake_units=("stake_units", "sum"), profit_units=("profit_units", "sum"))
        daily["roi_pct"] = 100 * daily.profit_units / daily.stake_units
        daily["cumulative_profit_units"] = daily.profit_units.cumsum()
        daily.to_csv(folder / "simulated_daily_results.csv")
        display(daily.round(3))
        print("Total simulated profit:", round(settled.profit_units.sum(), 3), "units | ROI:",
              round(100 * settled.profit_units.sum() / settled.stake_units.sum(), 2), "%")
    else:
        print("Simulated selections will be graded after final results on a later Run all.")
    return result

paper_results = run_paper_tracking(value_board, rankings, graded_predictions)
